In [ ]:
from langchain_ollama.llms import OllamaLLM 

model = OllamaLLM(model="gemma4:latest")
model.invoke('하늘이?')

In [ ]:
from langchain_core.messages import HumanMessage 
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(model="gemma4:latest")
prompt = [HumanMessage('フランスの首都はどこですか')]
result = model.invoke(prompt)
result

#### 각 역할에 따른 형태의 채팅 메시지 인터페이스 이용 
- HumanMessage : 사용자의 역할인 인간의 관점으로 작성한 메시지 
- AIMessage : 어시스턴트 역할인 AI 관점으로 작성한 메시지 
- SystemMessage : 시스템 역할인 AI가 준수할 지침을 설정하는 메시지 
- ChatMessage : 임의의 역할을 설정하는 메시지 

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage 
from langchain_ollama.chat_models import ChatOllama 

model = ChatOllama(model="gemma4:latest")

system_msg = SystemMessage('''당신은 문장 끝에 느낌표를 세개 붙여 대답하는 친절한 어시스턴트입니다.''')
human_msg = HumanMessage('フランスの首都はどこですか')

response = model.invoke([system_msg, human_msg])
print(response)

#### Prompt Template 적용

In [ ]:
from langchain_core.prompts import PromptTemplate 

template = PromptTemplate.from_template('''아래 작성한 컨텍스트를 기반으로 질문(Question)에 대답하세요.
    제공된 정보로 대답할 수 없는 질문이라면 "모르겠어요"라고 답하세요.
Context: {context} 
Question: {question}
Answer :
''')
prompt = template.invoke({
    'context' : '''거대 언어 모델은 자연어 처리 분야의 최신 발전을 이끌고 있습니다. 거대 언어 모델은 더 작은 모델보다 우수한 성능을 보이며 NLP 기능을 갖춘 애플리케이션을 개발하는 개발자들에게 매우 중요한 도구가 되었습니다. 
    개발자들은 Hugging Face의 'transfomers'라이브러리를 활용하거나, 'openai'및 'cohere'라이브러리를 통해 OpenAI와 Cohere의 서비스를 이용하여 거대 언어 모델을 활용할 수 있습니다.''',
    'question' : '거대 모델은 어디서 제공하나요?'
})

completion = model.invoke(prompt)

print(completion)


#### JSON 출력


In [ ]:
%pip install pydantic

In [ ]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel 

class AnswerWithJustification(BaseModel):
    '''사용자의 질문에 대한 답변과 그에 대한 근거(justification)를 함께 제공하세요'''
    answer: str
    '''사용자의 질문에 대한 답번'''
    justificaton: str
    '''답변에 대한 근거'''
llm = ChatOllama(model="gemma4:latest", temperature=0)
structured_llm = llm.with_structured_output(AnswerWithJustification)

result = structured_llm.invoke('''1킬로그램의 벽돌과 1킬로 그램의 깃털 중 어느 것이 더 무겁나요?''')
print(result.model_dump_json())

#### 기계가 바로 이해할 수 있는 형식과 출력 파서 


In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser 

parser = CommaSeparatedListOutputParser()
items = parser.invoke("apple, banana, cherry")

items

#### Runnable 인터페이스 
- 공통 인터페이스를 사용한다. 
- invoke : 하나의 입력을 하나의 출력으로 변환한다. 
- batch : 여러 입력을 여러 출력으로 변환한다. 
- stream : 하나의 입력이 생성하는 출력결과를 실시간으로 전달한다. 

In [ ]:
from langchain_ollama.chat_models import ChatOllama

completion = model.invoke('반갑습니다')
print(completion)

completions = model.batch(['반가워요','잘 있어요'])
print(completions)

for t in model.stream('잘 있어요'):
    print(t)

#### 명령형 구성 


In [ ]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain 

template = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 친절한 어시스턴트입니다'),
        ('human', '{question}')
    ]
)

model = ChatOllama(model='gemma4:latest')

@chain
def chatbot(val : ChatPromptTemplate):
    prompt = template.invoke(val)
    return model.invoke(prompt)

response = chatbot.invoke({'question':'거대 언어 모델은 어디에서 제공하나요?'})
print(response)

#### 선언형 구성

In [ ]:
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain 

template = ChatPromptTemplate.from_messages(
    [
        ('system', '당신은 친절한 어시스턴트입니다'),
        ('human', '{question}')
    ]
)

model = ChatOllama(model='gemma4:latest')

chatbot = template | model 

response = chatbot.invoke({'question':'거대 언어 모델은 어디에서 제공하나요?'})
print(response)